# Stage 01 — Carregamento dos Dados de Fala Interna e Obtenção das Épocas

Objetivo: selecionar somente os ensaios de **fala interna** e recortar o intervalo desejado do action interval

Execute as células na ordem. A última célula é a única que processa e salva todas as sessões.


## 1. Importações

- `Path`: manipulação de caminhos;
- `pickle`: leitura da matriz de eventos;
- `mne`: leitura e filtragem do EEG;
- `numpy`: organização e salvamento dos arrays.


In [352]:
from pathlib import Path
import pickle

import mne
import numpy as np


## 2. Caminhos e parâmetros

As bandas são guardadas em um dicionário para deixar explícito o significado de cada índice do tensor final.


In [353]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "thinking_outloud_dataset" / "derivatives"
OUTPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_01_load_and_epoching"

EPOCH_START_SECONDS = 1.0
EPOCH_END_SECONDS = 3.5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Entrada:", INPUT_DIR)
print("Saída:", OUTPUT_DIR)


Entrada: /home/jobson/Documentos/DSP_classifier/DSP-classifier/thinking_outloud_dataset/derivatives
Saída: /home/jobson/Documentos/DSP_classifier/DSP-classifier/processed_data/stage_01_load_and_epoching


## 3. Descobrir as sessões disponíveis

Em vez de fixar participantes e sessões no código, procuramos os arquivos `.fif` existentes.


In [354]:
epoch_files = sorted(INPUT_DIR.glob("sub-*/ses-*/*_eeg-epo.fif"))

print(f"Sessões encontradas: {len(epoch_files)}")
for path in epoch_files[:6]:
    print(" -", path.relative_to(PROJECT_ROOT))


Sessões encontradas: 30
 - thinking_outloud_dataset/derivatives/sub-01/ses-01/sub-01_ses-01_eeg-epo.fif
 - thinking_outloud_dataset/derivatives/sub-01/ses-02/sub-01_ses-02_eeg-epo.fif
 - thinking_outloud_dataset/derivatives/sub-01/ses-03/sub-01_ses-03_eeg-epo.fif
 - thinking_outloud_dataset/derivatives/sub-02/ses-01/sub-02_ses-01_eeg-epo.fif
 - thinking_outloud_dataset/derivatives/sub-02/ses-02/sub-02_ses-02_eeg-epo.fif
 - thinking_outloud_dataset/derivatives/sub-02/ses-03/sub-02_ses-03_eeg-epo.fif


## 4. Entender os eventos de uma sessão

Dados do Evento

Cada arquivo de dados de events.dat contém uma matriz de quatro colunas, onde cada linha corresponde a uma tentativa (trial).

A Coluna 0 contém o número da amostra em que o evento ocorreu.

A Coluna 1 contém a direção da palavra:

    0 = Arriba (para cima)
    1 = Abajo (para baixo)
    2 = Derecha (direita)
    3 = Izquierda (esquerda)

A Coluna 2 contém a condição da tentativa:

    0 = Fala pronunciada (fala aberta)
    1 = Fala interna (fala pensada)
    2 = Condição visualizada

A Coluna 3 contém o número da sessão:

    1 = Sessão 1
    2 = Sessão 2
    3 = Sessão 3



In [355]:
example_epochs_path = epoch_files[0]
example_name = example_epochs_path.name.replace("_eeg-epo.fif", "")
example_events_path = example_epochs_path.with_name(f"{example_name}_events.dat")

with example_events_path.open("rb") as file:
    example_events = pickle.load(file)

print("Formato dos eventos:", example_events.shape)
print("Primeiras linhas:\n", example_events[25:55])
print("Condições e quantidades:", np.unique(example_events[:, 2], return_counts=True))


Formato dos eventos: (200, 4)
Primeiras linhas:
 [[205582      2      0      1]
 [212169      1      0      1]
 [218723      0      0      1]
 [225583      2      0      1]
 [232154      2      0      1]
 [238605      3      0      1]
 [245431      0      0      1]
 [252309      1      0      1]
 [258914      2      0      1]
 [265348      2      0      1]
 [271833      2      0      1]
 [278420      3      0      1]
 [285110      0      0      1]
 [291630      0      0      1]
 [298303      1      0      1]
 [351788      1      1      1]
 [358546      1      1      1]
 [365390      3      1      1]
 [372216      3      1      1]
 [387985      3      1      1]
 [394471      0      1      1]
 [400870      2      1      1]
 [407629      3      1      1]
 [421964      3      1      1]
 [428808      1      1      1]
 [435481      0      1      1]
 [448707      1      1      1]
 [455482      3      1      1]
 [462138      3      1      1]
 [468606      1      1      1]]
Condições e quantida

## 5. Função que processa uma sessão

A função retorna os dados, os rótulos e informações úteis para conferência. Nenhum arquivo é salvo dentro dela.


In [ ]:
from IPython.display import display, HTML

# Define o caminho das suas quatro imagens
img1_path = './config/cap_128_front.png'
img2_path = './config/cap_128_back.png'
img3_path = './config/cap_128_left.png'
img4_path = './config/cap_128_right.png'

img5_path = './config/cap_128_layout_medium.jpg'



# Cria a estrutura usando tags <img> normais do HTML
html_code = f"""
<div style="display: flex; gap: 10px;">
    <div><img src="{img1_path}" width="400" /></div>
    <div><img src="{img2_path}" width="400" /></div>
    <div><img src="{img3_path}" width="400" /></div>
    <div><img src="{img4_path}" width="400" /></div>
</div>

<div style="display: flex; gap: 10px;">
    <img src="{img5_path}" width="1000" />
</div>

"""

# Exibe o resultado na tela
display(HTML(html_code))


# CHANNELS_OF_INTEREST = [

#     #--- Cortéx esquerdo ---

#     #Área de Broca
#     'D5', 'D6', 'D7',      'D8','D9', 'D10',      'D4', 'D11',
#     'C30', 'C31', 'C32', 

#     #Área de Wernicke
#     'D22', 'D23', 'D24', 'D25',

#     #Área Motora da Fala
#     'D12', #D10 já presente
#     'D19', 'D21', 

#     #Área Motora Suplementar (SMA)
#     'A1', 'C21', 'C22', 'C23',

#     #Córtex Pré-frontal
#     'C26', 'C27', 'C28', 'C29',

#     #--- Córtex direito ---

#     #Área de Broca
#     'C5', 'C6', 'C7',      'B27', 'B28','B29',     'C4', 'B30',
#     'C8', 'C9', 'C10',

#     #Área de Wernicke
#     'B25', 'B26', 'B14', 'B15',

#     #Área Motora da Fala
#     'B31', #'B29',
#     'B22', 'B24',

#     #Área Motora Suplementar (SMA)
#     #'A1', 'C21', 'C22', 'C23', já presente no lado esquerdo 

#     #Córtex Pré-frontal
#     'C13', 'C14', 'C15', 'C16',


# ]


CHANNELS_OF_INTEREST = [
    'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 
    'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 
    
    'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11', 'B12', 'B13', 'B14', 'B15', 'B16', 'B17', 'B18', 'B19', 'B20', 
    'B21', 'B22', 'B23', 'B24', 'B25', 'B26', 'B27', 'B28', 'B29', 'B30', 'B31', 'B32', 
    
    'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 
    'C21', 'C22', 'C23', 'C24', 'C25', 'C26', 'C27', 'C28', 'C29', 'C30', 'C31', 'C32', 
    
    'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'D16', 'D17', 'D18', 'D19', 'D20', 
    'D21', 'D22', 'D23', 'D24', 'D25', 'D26', 'D27', 'D28', 'D29', 'D30', 'D31', 'D32']




In [ ]:

def process_session(epochs_path):

    session_name = epochs_path.name.replace("_eeg-epo.fif", "")
    events_path = epochs_path.with_name(f"{session_name}_events.dat")

    epochs = mne.read_epochs(
        epochs_path,
        preload=True,
        verbose=False
    )

    with events_path.open("rb") as file:
        events = pickle.load(file)

    inner_speech_mask = events[:, 2] == 2 # 0 - pronounced speech , 1 - inner speech, 2 - visualized condition 
    inner_epochs = epochs[inner_speech_mask]

    inner_epochs.pick(CHANNELS_OF_INTEREST)

    labels = events[inner_speech_mask, 1].astype(int)
    sampling_rate = float(inner_epochs.info["sfreq"])

    start_sample = round(EPOCH_START_SECONDS * sampling_rate)
    end_sample = round(EPOCH_END_SECONDS * sampling_rate)

    data = inner_epochs.get_data()
    data = data[:, :, start_sample:end_sample]

    return (
        session_name,
        data,
        labels,
        inner_epochs.ch_names,
        sampling_rate,
    )

## 6. Testar com uma sessão

Esta célula permite conferir dimensões e classes antes de processar tudo.


In [358]:
name, data, labels, channel_names, sampling_rate = process_session(epoch_files[0])

print("Sessão:", name)
print("Taxa de amostragem:", sampling_rate)
print("Dados (épocas, canais, amostras):", data.shape)
print("Rótulos:", labels.shape)
print("Classes:", np.unique(labels, return_counts=True))
print("Numero de canais:", len(channel_names))
print("Canais:", channel_names[:])


expected_samples = round((EPOCH_END_SECONDS - EPOCH_START_SECONDS) * sampling_rate)
print(f"Expected samples: {expected_samples}")
print(f"Actual samples  : {data.shape[2]}")


Sessão: sub-01_ses-01
Taxa de amostragem: 256.0
Dados (épocas, canais, amostras): (40, 107, 640)
Rótulos: (40,)
Classes: (array([0, 1, 2, 3]), array([10, 10, 10, 10]))
Numero de canais: 107
Canais: ['A1', 'A2', 'A3', 'A4', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11', 'B12', 'B13', 'B14', 'B15', 'B16', 'B17', 'B18', 'B19', 'B20', 'B21', 'B22', 'B23', 'B24', 'B25', 'B26', 'B27', 'B28', 'B29', 'B30', 'B31', 'B32', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24', 'C25', 'C26', 'C27', 'C28', 'C29', 'C30', 'C31', 'C32', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'D16', 'D17', 'D18', 'D19', 'D20', 'D21', 'D22', 'D23', 'D24', 'D25', 'D26', 'D27', 'D28', 'D29', 'D30', 'D31', 'D32']
Expected samples: 640
Actual samples  : 640


## 7. Processar e salvar todas as sessões

Execute esta célula somente depois de conferir o exemplo acima.


In [359]:
for epochs_path in epoch_files:
    name, data, labels, _, _ = process_session(epochs_path)

    np.save(OUTPUT_DIR / f"{name}_inner_epochs.npy", data)
    np.save(OUTPUT_DIR / f"{name}_inner_epochs_labels.npy", labels)

    print(f"{name}: dados={data.shape}, rótulos={labels.shape}")

print("Stage 01 finalizado.")


sub-01_ses-01: dados=(40, 107, 640), rótulos=(40,)
sub-01_ses-02: dados=(40, 107, 640), rótulos=(40,)
sub-01_ses-03: dados=(20, 107, 640), rótulos=(20,)
sub-02_ses-01: dados=(40, 107, 640), rótulos=(40,)
sub-02_ses-02: dados=(40, 107, 640), rótulos=(40,)
sub-02_ses-03: dados=(40, 107, 640), rótulos=(40,)
sub-03_ses-01: dados=(40, 107, 640), rótulos=(40,)
sub-03_ses-02: dados=(40, 107, 640), rótulos=(40,)
sub-03_ses-03: dados=(20, 107, 640), rótulos=(20,)
sub-04_ses-01: dados=(40, 107, 640), rótulos=(40,)
sub-04_ses-02: dados=(40, 107, 640), rótulos=(40,)
sub-04_ses-03: dados=(40, 107, 640), rótulos=(40,)
sub-05_ses-01: dados=(40, 107, 640), rótulos=(40,)
sub-05_ses-02: dados=(40, 107, 640), rótulos=(40,)
sub-05_ses-03: dados=(40, 107, 640), rótulos=(40,)
sub-06_ses-01: dados=(40, 107, 640), rótulos=(40,)
sub-06_ses-02: dados=(40, 107, 640), rótulos=(40,)
sub-06_ses-03: dados=(28, 107, 640), rótulos=(28,)
sub-07_ses-01: dados=(40, 107, 640), rótulos=(40,)
sub-07_ses-02: dados=(40, 107, 